In [1]:
import os
import shutil
import sys
import subprocess

In [2]:
program_path = "../src/GenerateDataset/build"

# Dataset Path
dest_dataset_path = "../data/2000_Plots_20241210_BetterQuantized"

# Clean the dataset directory
if os.path.exists(dest_dataset_path):
    shutil.rmtree(dest_dataset_path)

# Create the dataset directory
if not os.path.exists(dest_dataset_path):
    os.makedirs(dest_dataset_path)

# Create images directory
images_path = dest_dataset_path + "/images"
if not os.path.exists(images_path):
    os.makedirs(images_path)

# Create xml files directory
xml_path = dest_dataset_path + "/xml"
if not os.path.exists(xml_path):
    os.makedirs(xml_path)

output_path = "output"
# Get absolute path
output_path = os.path.abspath(output_path)

In [7]:
n_iter = 100

# Remove and recreate output folder
shutil.rmtree(output_path, ignore_errors=True)
os.makedirs(output_path, exist_ok=True)


# Check the current platform
import platform
current_platform = platform.system()
# Set the accelerator based on the platform
if current_platform == "Darwin":
    pass
else:
    os.environ["DISPLAY"] = ":12.0"
    
# Generate dataset
for i in range(n_iter):
    print("Generating image: ", i)
    seed = i
    image_name = f"cowpea_{i:04d}" 
    # Generate image 
    # Construct the command
    command = ""
    command += f"cd {program_path} && ./main " 
    command += f"-h 1.0 -o {output_path} -seed {seed} -name {image_name} -xml -tile none -g"
    result = subprocess.run(command, shell=True, capture_output=True, text=True)


    # Check if the command was successful
    if result.returncode == 0:
        # print("Command executed successfully")
        print(result.stdout)  # Print the standard output
        pass
    else:
        print(result.stdout)  # Print the standard output
        print(result.stderr)  # Print the error output
        raise("Command failed")
        pass

Generating image:  0


Save dir: /home/lion397/codes/Image2PlantArchitecture/src/output
Seed: 0
Output name: cowpea_0000
Debug: false
Grow: true
View height: 1m
Tile file: none
Initializing graphics...done.
writing JPEG image: /home/lion397/codes/Image2PlantArchitecture/src/output/cowpea_0000.jpeg
writing JPEG image: /home/lion397/codes/Image2PlantArchitecture/src/output/cowpea_0000_day_00.jpeg
writing JPEG image: /home/lion397/codes/Image2PlantArchitecture/src/output/cowpea_0000_day_01.jpeg
writing JPEG image: /home/lion397/codes/Image2PlantArchitecture/src/output/cowpea_0000_day_02.jpeg
writing JPEG image: /home/lion397/codes/Image2PlantArchitecture/src/output/cowpea_0000_day_03.jpeg
writing JPEG image: /home/lion397/codes/Image2PlantArchitecture/src/output/cowpea_0000_day_04.jpeg
writing JPEG image: /home/lion397/codes/Image2PlantArchitecture/src/output/cowpea_0000_day_05.jpeg
writing JPEG image: /home/lion397/codes/Image2PlantArchitecture/src/output/cowpea_0000_day_06.jpeg
writing JPEG image: /home/lion3

In [8]:
import os
import subprocess
import concurrent.futures
from tqdm import tqdm

# Function to re-render a single XML file
def re_render_xml(filename, rotation=True):
    if filename.endswith(".xml"):
        image_name = filename.split(".")[0]
        # Generate image 
        # Construct the command
        command = ""
        command += f"cd {program_path} && ./main " 
        command += f"-h 1.0 -o {output_path} -name {image_name} -tile none -f {os.path.join(output_path, filename)}"
        if rotation:
            command += " -r"
        result = subprocess.run(command, shell=True, capture_output=True, text=True)
        return result

# Re-render XML files in parallel with progress bar
with concurrent.futures.ThreadPoolExecutor() as executor:
    xml_list = os.listdir(output_path)
    xml_list.sort()
    futures = [executor.submit(re_render_xml, filename) for filename in xml_list if filename.endswith(".xml")]
    
    # Use tqdm to show progress bar
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Re-rendering XML files"):
        result = future.result()
        # Check if the command was successful
        if result.returncode != 0:
            print(result.stdout)  # Print the standard output
            print(result.stderr)  # Print the error output
            raise Exception("Command failed")

Re-rendering XML files: 100%|██████████| 2000/2000 [08:49<00:00,  3.78it/s]


In [9]:
# List jpg and xml files and move them to the dataset directory
for filename in os.listdir(output_path):
    if filename.endswith(".jpeg"):
        shutil.move(os.path.join(output_path, filename), images_path)
    elif filename.endswith(".xml"):
        shutil.move(os.path.join(output_path, filename), xml_path)

# Quantize XML and genreate dataset again

In [2]:
import os
import shutil
import sys
import subprocess

# Quantize XML and re-render
program_path = "../src/GenerateDataset/build"

# Dataset Path
source_dataset = "../data/2000_Plots_20241210"
dest_dataset_path = "../data/2000_Plots_20241210_BetterQuantized"


# Create the dataset directory
if not os.path.exists(dest_dataset_path):
    os.makedirs(dest_dataset_path)

# Create images directory
images_path = dest_dataset_path + "/images"
if not os.path.exists(images_path):
    os.makedirs(images_path)

# Create xml files directory
xml_path = dest_dataset_path + "/xml"
if not os.path.exists(xml_path):
    os.makedirs(xml_path)

output_path = "output"
# Get absolute path
output_path = os.path.abspath(output_path)

In [3]:
# Read all xml and quantize
import xml.etree.ElementTree as ET
from string_to_xml_to_vec import xml2vec, pretty_print_xml, linked_to_recursive
from string_to_xml_to_vec import vec2xml, recursive_to_linked
from plant_tokenizer import token2vec, vec2token
from tqdm.auto import tqdm
xml_path = os.path.join(source_dataset, "xml")
xml_list = os.listdir(xml_path)
xml_list = [filename for filename in xml_list if filename.endswith(".xml")]
xml_list.sort()

for xml_name in tqdm(xml_list):
    # Read XML
    xml_file = os.path.join(xml_path, xml_name)
    tree = ET.parse(xml_file)
    root = tree.getroot()
    root = linked_to_recursive(root)
    for plant_instance in root:
        plant_instance_array = []
        xml2vec(plant_instance, plant_instance_array)        
    
    # Quantize
    tokens = vec2token(plant_instance_array)

    # Save XML
    plant_vec = token2vec(tokens)
    plant_xml = vec2xml(plant_vec)
    plant_xml_file_name = f"{output_path}/{xml_name}"
    plant_xml = recursive_to_linked(plant_xml)
    plant_xml_str = pretty_print_xml(plant_xml)
    with open(plant_xml_file_name, "w") as f:
        f.write(plant_xml_str)

  0%|          | 0/40000 [00:00<?, ?it/s]

In [5]:
import os
import subprocess
import concurrent.futures
from tqdm import tqdm

# Check the current platform
import platform
current_platform = platform.system()
# Set the accelerator based on the platform
if current_platform == "Darwin":
    pass
else:
    os.environ["DISPLAY"] = ":11.0"
    
# Function to re-render a single XML file
def re_render_xml(filename, rotation=True):
    if filename.endswith(".xml"):
        image_name = filename.split(".")[0]
        # Generate image 
        # Construct the command
        command = ""
        command += f"cd {program_path} && ./main " 
        command += f"-h 1.0 -o {output_path} -name {image_name} -tile none -f {os.path.join(output_path, filename)}"
        if rotation:
            command += " -r"
        result = subprocess.run(command, shell=True, capture_output=True, text=True)
        return result

# Re-render XML files in parallel with progress bar
with concurrent.futures.ThreadPoolExecutor() as executor:
    xml_list = os.listdir(output_path)
    xml_list.sort()
    futures = [executor.submit(re_render_xml, filename) for filename in xml_list if filename.endswith(".xml")]
    
    # Use tqdm to show progress bar
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Re-rendering XML files"):
        result = future.result()
        # Check if the command was successful
        if result.returncode != 0:
            print(result.stdout)  # Print the standard output
            print(result.stderr)  # Print the error output
            raise Exception("Command failed")

Re-rendering XML files: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 40000/40000 [2:26:37<00:00,  4.55it/s]


In [6]:
# List jpg and xml files and move them to the dataset directory
images_path = dest_dataset_path + "/images"
xml_path = dest_dataset_path + "/xml"
# output_path, images_path, xml_path 변수가 이미 정의되어 있다고 가정합니다.
for filename in os.listdir(output_path):
    # 이미지 파일 처리
    if filename.endswith(".jpeg") or filename.endswith(".png"):  # 원하는 이미지 확장자를 사용합니다.
        src_path = os.path.join(output_path, filename)
        dst_path = os.path.join(images_path, filename)
        
        # 파일이 이미 존재하는 경우 삭제
        if os.path.exists(dst_path):
            os.remove(dst_path)

        shutil.move(src_path, dst_path)
    
    # XML 파일 처리
    elif filename.endswith(".xml"):
        src_path = os.path.join(output_path, filename)
        dst_path = os.path.join(xml_path, filename)
        
        # 파일이 이미 존재하는 경우 삭제
        if os.path.exists(dst_path):
            os.remove(dst_path)

        shutil.move(src_path, dst_path)